In [ ]:
# Making Pickel file for GloVe embedding to improve latence from .vec file
import numpy as np
import pickle

glove_embeddings_path = r"C:\Users\gaurm\OneDrive\Desktop\Glove_Word_Embeddings_200D\wiki_giga_2024_200_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt"
save_path = r"C:\Users\gaurm\OneDrive\Desktop\glove_embeddings_200d.pkl"

embeddings_dict = {}
expected_dim = 200 

with open(glove_embeddings_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        if len(values) != expected_dim + 1: 
            continue 
        
        try:
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings_dict[word] = vector
        except ValueError:
            continue

with open(save_path, 'wb') as f:
    pickle.dump(embeddings_dict, f)

print(f"Cleaned dictionary saved with {len(embeddings_dict)} valid vectors.")

In [ ]:
#King Queen  test to check quality of embeddings
import pickle
import numpy as np


with open(r"C:\Users\gaurm\OneDrive\Desktop\glove_embeddings_200d.pkl", 'rb') as f:
    embeddings_dict = pickle.load(f)


words = list(embeddings_dict.keys())

matrix = np.array(list(embeddings_dict.values())) 


norms = np.linalg.norm(matrix, axis=1, keepdims=True)
matrix_normed = matrix / norms

def find_closest_matrix(target_vec, top_n=5):
    target_normed = target_vec / np.linalg.norm(target_vec)
    
    similarities = np.dot(matrix_normed, target_normed)
    
   
    best_indices = np.argsort(similarities)[-top_n:][::-1]
    
    return [(words[i], similarities[i]) for i in best_indices]


target = embeddings_dict['king'] - embeddings_dict['man'] + embeddings_dict['woman']
results = find_closest_matrix(target)

print("Top results:")
for word, score in results:
    print(f"{word}: {score:.4f}")


In [ ]:
#Fastext Trainign Loop + Pre Processing
import json
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

import fasttext  


TRAIN_PATH = "/kaggle/input/datasets/madhavgaur12321/dataset-train-val/train_data.jsonl"
VAL_PATH   = "/kaggle/input/datasets/madhavgaur12321/dataset-train-val/val_data.jsonl"


FASTTEXT_BIN_PATH = "/kaggle/input/datasets/madhavgaur12321/fast-text-embeddings/crawl-300d-2M-subword.bin"

k = 15
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-3
MAX_VOCAB = 200000
MIN_FREQ = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


_num_re = re.compile(r"^\d+([.,]\d+)?$")
def norm_tok(t: str) -> str:
    t = t.strip()
    if _num_re.match(t):
        return "<num>"
    return t.lower()


def read_jsonl(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            toks = [norm_tok(t) for t in ex["tokens"]]
            labs = ex["labels"]
            if len(toks) != len(labs):
                raise ValueError(f"tokens/labels length mismatch in {path}")
            items.append((toks, labs))
    return items

train_data = read_jsonl(TRAIN_PATH)
val_data   = read_jsonl(VAL_PATH)


PAD, UNK = "<pad>", "<unk>"
LAB_PAD  = "<pad>"

tok_counts = Counter(t for toks, _ in train_data for t in toks)
vocab_tokens = [t for t, c in tok_counts.most_common(MAX_VOCAB) if c >= MIN_FREQ]

itos_w = [PAD, UNK] + [t for t in vocab_tokens if t not in (PAD, UNK)]
stoi = {w: i for i, w in enumerate(itos_w)}

label_set = sorted({lab for _, labs in train_data for lab in labs})
itos_l = [LAB_PAD] + label_set
ltoi = {l: i for i, l in enumerate(itos_l)}

def encode_tokens(toks):
    return [stoi.get(t, stoi[UNK]) for t in toks]

def encode_labels(labs):
    return [ltoi[l] for l in labs]


class TagDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        toks, labs = self.data[idx]
        x = torch.tensor(encode_tokens(toks), dtype=torch.long)
        y = torch.tensor(encode_labels(labs), dtype=torch.long)
        return x, y

def collate(batch):
    xs, ys = zip(*batch)
    lens = torch.tensor([len(x) for x in xs], dtype=torch.long)
    maxlen = int(lens.max())

    xpad = torch.full((len(xs), maxlen), stoi[PAD], dtype=torch.long)
    ypad = torch.full((len(xs), maxlen), ltoi[LAB_PAD], dtype=torch.long)

    for i, (x, y) in enumerate(zip(xs, ys)):
        xpad[i, :len(x)] = x
        ypad[i, :len(y)] = y

    return xpad, ypad, lens

train_loader = DataLoader(TagDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(TagDataset(val_data),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


ft = fasttext.load_model(FASTTEXT_BIN_PATH)
EMB_DIM = ft.get_dimension()
print(f"fastText loaded | dim={EMB_DIM}")

emb_mat = np.random.normal(0, 0.1, size=(len(stoi), EMB_DIM)).astype(np.float32)
emb_mat[stoi[PAD]] = 0.0

for w, idx in stoi.items():
    if w == PAD:
        continue
    emb_mat[idx] = ft.get_word_vector(w)

emb_init = torch.tensor(emb_mat, dtype=torch.float32)


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_k, num_labels, pad_idx, emb_init_tensor):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb.weight.data.copy_(emb_init_tensor)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_k,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(2 * hidden_k, num_labels)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        out_packed, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True)
        logits = self.fc(out)
        return logits

model = BiLSTMTagger(
    vocab_size=len(stoi),
    emb_dim=EMB_DIM,
    hidden_k=k,
    num_labels=len(itos_l),
    pad_idx=stoi[PAD],
    emb_init_tensor=emb_init
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=ltoi[LAB_PAD])
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def extract_entities(labels):
    """
    Given a list of BIO labels (no padding),
    return a set of (start_idx, end_idx, entity_type).
    end_idx is exclusive.
    """
    entities = set()
    i = 0
    while i < len(labels):
        if labels[i].startswith("B-"):
            etype = labels[i][2:]
            start = i
            i += 1
            while i < len(labels) and labels[i] == f"I-{etype}":
                i += 1
            entities.add((start, i, etype))
        else:
            i += 1
    return entities


def free_match_entities(pred_entities, gold_entities):
    """
    Returns (tp, fp, fn) using overlap + same-type matching, 1-to-1.
    """
    pred_list = list(pred_entities)
    gold_list = list(gold_entities)

    matched_gold = set()
    tp = 0

    for sp, ep, tp_type in pred_list:
        for gi, (sg, eg, tg_type) in enumerate(gold_list):
            if gi in matched_gold:
                continue
            # same type AND overlapping spans
            if tp_type == tg_type and max(sp, sg) < min(ep, eg):
                matched_gold.add(gi)
                tp += 1
                break  # this pred is matched, move to next pred

    fp = len(pred_list) - tp
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn


def evaluate():
    model.eval()
    total, correct = 0, 0

    C = len(itos_l)
    tp = np.zeros(C, dtype=np.int64)
    fp = np.zeros(C, dtype=np.int64)
    fn = np.zeros(C, dtype=np.int64)

    
    token_entity_tp = 0
    token_entity_fp = 0
    token_entity_fn = 0

    total_sentences = 0
    exact_matches = 0

    with torch.no_grad():
        for x, y, lens in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x, lens)
            pred = logits.argmax(-1)

            mask = (y != ltoi[LAB_PAD])
            total += int(mask.sum().item())
            correct += int(((pred == y) & mask).sum().item())

            for c in range(1, C):
                pc = (pred == c) & mask
                yc = (y == c) & mask
                tp[c] += int((pc & yc).sum().item())
                fp[c] += int((pc & ~yc).sum().item())
                fn[c] += int((~pc & yc).sum().item())

            for i in range(len(lens)):
                L = int(lens[i])
                pred_labels = [itos_l[int(p)] for p in pred[i, :L]]
                gold_labels = [itos_l[int(g)] for g in y[i, :L]]

                total_sentences += 1

                if pred_labels == gold_labels:
                    exact_matches += 1

               
                for pred_lab, gold_lab in zip(pred_labels, gold_labels):
                    pred_is_entity = pred_lab.startswith("B-") or pred_lab.startswith("I-")
                    gold_is_entity = gold_lab.startswith("B-") or gold_lab.startswith("I-")

                    # Extract entity type (e.g., "PER" from "B-PER")
                    pred_type = pred_lab[2:] if pred_is_entity else None
                    gold_type = gold_lab[2:] if gold_is_entity else None

                    if pred_is_entity and gold_is_entity:
                        if pred_type == gold_type:
                            token_entity_tp += 1  # Both entity, same type
                        else:
                            token_entity_fp += 1  # Pred entity, wrong type
                            token_entity_fn += 1  # Missed gold entity
                    elif pred_is_entity and not gold_is_entity:
                        token_entity_fp += 1  # Predicted entity where there's none
                    elif not pred_is_entity and gold_is_entity:
                        token_entity_fn += 1  # Missed an entity token

    acc = correct / max(total, 1)

    f1s = []
    for c in range(1, C):
        p = tp[c] / max(tp[c] + fp[c], 1)
        r = tp[c] / max(tp[c] + fn[c], 1)
        f1 = 0.0 if (p + r) == 0 else (2 * p * r / (p + r))
        f1s.append(f1)
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0

    # Token-level FreeMatch F1
    ent_p = token_entity_tp / max(token_entity_tp + token_entity_fp, 1)
    ent_r = token_entity_tp / max(token_entity_tp + token_entity_fn, 1)
    freematch_f1 = 0.0 if (ent_p + ent_r) == 0 else (2 * ent_p * ent_r / (ent_p + ent_r))

    strict_em = exact_matches / max(total_sentences, 1)

    return acc, macro_f1, freematch_f1, strict_em


history = {
    "train_loss": [],
    "val_acc": [],
    "val_macroF1": [],
    "val_FreeMatchF1": [],
    "val_StrictEM": [],
}


for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0

    for x, y, lens in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        logits = model(x, lens)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        running += float(loss.item())

    avg_train_loss = running / len(train_loader)
    val_acc, val_f1, val_freematch_f1, val_strict_em = evaluate()

    # Store for plotting
    history["train_loss"].append(avg_train_loss)
    history["val_acc"].append(val_acc)
    history["val_macroF1"].append(val_f1)
    history["val_FreeMatchF1"].append(val_freematch_f1)
    history["val_StrictEM"].append(val_strict_em)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_macroF1={val_f1:.4f} | "
        f"val_FreeMatchF1={val_freematch_f1:.4f} | "
        f"val_StrictEM={val_strict_em:.4f}"
    )


epochs_range = list(range(1, EPOCHS + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"BiLSTM Tagger — fastText 300d — k={2}", fontsize=15, fontweight="bold")


ax = axes[0, 0]
ax.plot(epochs_range, history["train_loss"], "o-", color="red", linewidth=2, markersize=5)
ax.set_title("Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)


ax = axes[0, 1]
ax.plot(epochs_range, history["val_acc"], "o-", color="blue", linewidth=2, markersize=5, label="Val Accuracy")
ax.plot(epochs_range, history["val_macroF1"], "s-", color="green", linewidth=2, markersize=5, label="Val Macro-F1")
ax.set_title("Val Accuracy & Macro-F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(epochs_range, history["val_FreeMatchF1"], "D-", color="purple", linewidth=2, markersize=5)
ax.set_title("Val FreeMatch F1 (overlap + same type)")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1")
ax.grid(True, alpha=0.3)


ax = axes[1, 1]
ax.plot(epochs_range, history["val_StrictEM"], "^-", color="orange", linewidth=2, markersize=5)
ax.set_title("Val Strict Exact Match")
ax.set_xlabel("Epoch")
ax.set_ylabel("EM")
ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f"training_curves_fasttext_k{k}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved: training_curves_fasttext_k{k}.png")

OUT_PATH = f"bilstm_biotag_fasttext_k{k}.pt"

torch.save(
    {
        "state_dict": model.state_dict(),
        "stoi": stoi,
        "itos_w": itos_w,
        "ltoi": ltoi,
        "itos_l": itos_l,
        "emb_dim": EMB_DIM,
        "hidden_k": k,
    },
    OUT_PATH,
)
print("Saved:", OUT_PATH)

In [ ]:
import json
import re
import pickle
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os

TRAIN_PATH = "/kaggle/input/datasets/madhavgaur12321/dataset-train-val/train_data.jsonl"
VAL_PATH   = "/kaggle/input/datasets/madhavgaur12321/dataset-train-val/val_data.jsonl"

GLOVE_PKL_PATH = r"C:\Users\gaurm\OneDrive\Desktop\glove_embeddings_200d.pkl"

k = 15
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-3
MAX_VOCAB = 200000
MIN_FREQ = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


_num_re = re.compile(r"^\d+([.,]\d+)?$")
def norm_tok(t: str) -> str:
    t = t.strip()
    if _num_re.match(t):
        return "<num>"
    return t.lower()


def read_jsonl(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            toks = [norm_tok(t) for t in ex["tokens"]]
            labs = ex["labels"]
            if len(toks) != len(labs):
                raise ValueError(f"tokens/labels length mismatch in {path}")
            items.append((toks, labs))
    return items

train_data = read_jsonl(TRAIN_PATH)
val_data   = read_jsonl(VAL_PATH)


PAD, UNK = "<pad>", "<unk>"
LAB_PAD  = "<pad>"

tok_counts = Counter(t for toks, _ in train_data for t in toks)
vocab_tokens = [t for t, c in tok_counts.most_common(MAX_VOCAB) if c >= MIN_FREQ]

itos_w = [PAD, UNK] + [t for t in vocab_tokens if t not in (PAD, UNK)]
stoi = {w: i for i, w in enumerate(itos_w)}

label_set = sorted({lab for _, labs in train_data for lab in labs})
itos_l = [LAB_PAD] + label_set
ltoi = {l: i for i, l in enumerate(itos_l)}

def encode_tokens(toks):
    return [stoi.get(t, stoi[UNK]) for t in toks]

def encode_labels(labs):
    return [ltoi[l] for l in labs]


class TagDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        toks, labs = self.data[idx]
        x = torch.tensor(encode_tokens(toks), dtype=torch.long)
        y = torch.tensor(encode_labels(labs), dtype=torch.long)
        return x, y

def collate(batch):
    xs, ys = zip(*batch)
    lens = torch.tensor([len(x) for x in xs], dtype=torch.long)
    maxlen = int(lens.max())

    xpad = torch.full((len(xs), maxlen), stoi[PAD], dtype=torch.long)
    ypad = torch.full((len(xs), maxlen), ltoi[LAB_PAD], dtype=torch.long)

    for i, (x, y) in enumerate(zip(xs, ys)):
        xpad[i, :len(x)] = x
        ypad[i, :len(y)] = y

    return xpad, ypad, lens

train_loader = DataLoader(TagDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(TagDataset(val_data),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

def load_glove_pickle(glove_pkl_path):
    with open(glove_pkl_path, "rb") as f:
        glove = pickle.load(f)
    if not isinstance(glove, dict):
        raise TypeError("GloVe pickle must be a dict: {word: vector}")
    any_key = next(iter(glove.keys()))
    if isinstance(any_key, bytes):
        glove = {k.decode("utf-8", errors="ignore"): v for k, v in glove.items()}
    any_vec = np.asarray(next(iter(glove.values())), dtype=np.float32)
    if any_vec.ndim != 1:
        raise ValueError("GloVe vectors must be 1D arrays")
    dim = int(any_vec.shape[0])
    return glove, dim

glove_dict, EMB_DIM = load_glove_pickle(GLOVE_PKL_PATH)
print(f"GloVe loaded: {len(glove_dict)} words | dim={EMB_DIM}")

emb_mat = np.random.normal(0, 0.1, size=(len(stoi), EMB_DIM)).astype(np.float32)
emb_mat[stoi[PAD]] = 0.0

found = 0
for w, idx in stoi.items():
    if w in (PAD, UNK):
        continue
    v = glove_dict.get(w)
    if v is None and w != w.lower():
        v = glove_dict.get(w.lower())
    if v is not None:
        emb_mat[idx] = np.asarray(v, dtype=np.float32)
        found += 1

print(f"Matched {found}/{len(stoi)} vocab tokens to GloVe")
emb_init = torch.tensor(emb_mat, dtype=torch.float32)


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_k, num_labels, pad_idx, emb_init_tensor):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb.weight.data.copy_(emb_init_tensor)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_k,
            num_layers=3,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(2 * hidden_k, num_labels)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        out_packed, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True)
        logits = self.fc(out)
        return logits

model = BiLSTMTagger(
    vocab_size=len(stoi),
    emb_dim=EMB_DIM,
    hidden_k=k,
    num_labels=len(itos_l),
    pad_idx=stoi[PAD],
    emb_init_tensor=emb_init
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=ltoi[LAB_PAD])
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def extract_entities(labels):
    """
    Given a list of BIO labels (no padding),
    return a set of (start_idx, end_idx, entity_type).
    end_idx is exclusive.
    """
    entities = set()
    i = 0
    while i < len(labels):
        if labels[i].startswith("B-"):
            etype = labels[i][2:]
            start = i
            i += 1
            while i < len(labels) and labels[i] == f"I-{etype}":
                i += 1
            entities.add((start, i, etype))
        else:
            i += 1
    return entities


def free_match_entities(pred_entities, gold_entities):
   
    pred_list = list(pred_entities)
    gold_list = list(gold_entities)

    matched_gold = set()
    tp = 0

    for sp, ep, tp_type in pred_list:
        for gi, (sg, eg, tg_type) in enumerate(gold_list):
            if gi in matched_gold:
                continue
            
            if tp_type == tg_type and max(sp, sg) < min(ep, eg):
                matched_gold.add(gi)
                tp += 1
                break  

    fp = len(pred_list) - tp
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn


def evaluate():
    model.eval()
    total, correct = 0, 0

    C = len(itos_l)
    tp = np.zeros(C, dtype=np.int64)
    fp = np.zeros(C, dtype=np.int64)
    fn = np.zeros(C, dtype=np.int64)

   
    token_entity_tp = 0
    token_entity_fp = 0
    token_entity_fn = 0

    total_sentences = 0
    exact_matches = 0

    with torch.no_grad():
        for x, y, lens in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x, lens)
            pred = logits.argmax(-1)

            mask = (y != ltoi[LAB_PAD])
            total += int(mask.sum().item())
            correct += int(((pred == y) & mask).sum().item())

            for c in range(1, C):
                pc = (pred == c) & mask
                yc = (y == c) & mask
                tp[c] += int((pc & yc).sum().item())
                fp[c] += int((pc & ~yc).sum().item())
                fn[c] += int((~pc & yc).sum().item())

            for i in range(len(lens)):
                L = int(lens[i])
                pred_labels = [itos_l[int(p)] for p in pred[i, :L]]
                gold_labels = [itos_l[int(g)] for g in y[i, :L]]

                total_sentences += 1

                if pred_labels == gold_labels:
                    exact_matches += 1

               
                for pred_lab, gold_lab in zip(pred_labels, gold_labels):
                    pred_is_entity = pred_lab.startswith("B-") or pred_lab.startswith("I-")
                    gold_is_entity = gold_lab.startswith("B-") or gold_lab.startswith("I-")

                    # Extract entity type (e.g., "PER" from "B-PER")
                    pred_type = pred_lab[2:] if pred_is_entity else None
                    gold_type = gold_lab[2:] if gold_is_entity else None

                    if pred_is_entity and gold_is_entity:
                        if pred_type == gold_type:
                            token_entity_tp += 1  # Both entity, same type
                        else:
                            token_entity_fp += 1  # Pred entity, wrong type
                            token_entity_fn += 1  # Missed gold entity
                    elif pred_is_entity and not gold_is_entity:
                        token_entity_fp += 1  # Predicted entity where there's none
                    elif not pred_is_entity and gold_is_entity:
                        token_entity_fn += 1  # Missed an entity token

    acc = correct / max(total, 1)

    f1s = []
    for c in range(1, C):
        p = tp[c] / max(tp[c] + fp[c], 1)
        r = tp[c] / max(tp[c] + fn[c], 1)
        f1 = 0.0 if (p + r) == 0 else (2 * p * r / (p + r))
        f1s.append(f1)
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0

    # Token-level FreeMatch F1
    ent_p = token_entity_tp / max(token_entity_tp + token_entity_fp, 1)
    ent_r = token_entity_tp / max(token_entity_tp + token_entity_fn, 1)
    freematch_f1 = 0.0 if (ent_p + ent_r) == 0 else (2 * ent_p * ent_r / (ent_p + ent_r))

    strict_em = exact_matches / max(total_sentences, 1)

    return acc, macro_f1, freematch_f1, strict_em


history = {
    "train_loss": [],
    "val_acc": [],
    "val_macroF1": [],
    "val_FreeMatchF1": [],
    "val_StrictEM": [],
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0

    for x, y, lens in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        logits = model(x, lens)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        running += float(loss.item())

    avg_train_loss = running / len(train_loader)
    val_acc, val_f1, val_freematch_f1, val_strict_em = evaluate()

    # Store for plotting
    history["train_loss"].append(avg_train_loss)
    history["val_acc"].append(val_acc)
    history["val_macroF1"].append(val_f1)
    history["val_FreeMatchF1"].append(val_freematch_f1)
    history["val_StrictEM"].append(val_strict_em)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"val_macroF1={val_f1:.4f} | "
        f"val_FreeMatchF1={val_freematch_f1:.4f} | "
        f"val_StrictEM={val_strict_em:.4f}"
    )


epochs_range = list(range(1, EPOCHS + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"BiLSTM Tagger — GloVe 200d — k={3}", fontsize=15, fontweight="bold")

# 1) Training Loss
ax = axes[0, 0]
ax.plot(epochs_range, history["train_loss"], "o-", color="red", linewidth=2, markersize=5)
ax.set_title("Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)


ax = axes[0, 1]
ax.plot(epochs_range, history["val_acc"], "o-", color="blue", linewidth=2, markersize=5, label="Val Accuracy")
ax.plot(epochs_range, history["val_macroF1"], "s-", color="green", linewidth=2, markersize=5, label="Val Macro-F1")
ax.set_title("Val Accuracy & Macro-F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True, alpha=0.3)


ax = axes[1, 0]
ax.plot(epochs_range, history["val_FreeMatchF1"], "D-", color="purple", linewidth=2, markersize=5)
ax.set_title("Val FreeMatch F1 (overlap + same type)")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1")
ax.grid(True, alpha=0.3)


ax = axes[1, 1]
ax.plot(epochs_range, history["val_StrictEM"], "^-", color="orange", linewidth=2, markersize=5)
ax.set_title("Val Strict Exact Match")
ax.set_xlabel("Epoch")
ax.set_ylabel("EM")
ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f"training_curves_glove_k{k}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved: training_curves_glove_k{k}.png")


OUT_PATH = f"bilstm_biotag_glove_k{k}.pt"

torch.save(
    {
        "state_dict": model.state_dict(),
        "stoi": stoi,
        "itos_w": itos_w,
        "ltoi": ltoi,
        "itos_l": itos_l,
        "emb_dim": EMB_DIM,
        "hidden_k": k,
    },
    OUT_PATH,
)
print("Saved:", OUT_PATH)

In [ ]:
#Load test JsonL writes the embedding in  the same format as train.jsonl 
import json
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


CKPT_PATH = r"C:\Users\gaurm\OneDrive\Desktop\bilstm_biotag_glovePkl_k6.pt"
TEST_PATH = r"C:\Users\gaurm\Downloads\dataset\dataset\val_data.jsonl"
OUT_PATH  = r"C:\Users\gaurm\OneDrive\Desktop\output_output.jsonl"

BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


_num_re = re.compile(r"^\d+([.,]\d+)?$")
def norm_tok(t: str) -> str:
    t = t.strip()
    if _num_re.match(t):
        return "<num>"
    return t.lower()


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_k, num_labels, pad_idx):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_k,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(2 * hidden_k, num_labels)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            e, lens.cpu(), batch_first=True, enforce_sorted=False
        )
        out_packed, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True)
        logits = self.fc(out)
        return logits


ckpt = torch.load(CKPT_PATH, map_location="cpu")

stoi   = ckpt["stoi"]
itos_l = ckpt["itos_l"]
emb_dim = ckpt["emb_dim"]
hidden_k = ckpt["hidden_k"]

PAD, UNK = "<pad>", "<unk>"
pad_idx = stoi[PAD]
unk_idx = stoi[UNK]

model = BiLSTMTagger(
    vocab_size=len(stoi),
    emb_dim=emb_dim,
    hidden_k=hidden_k,
    num_labels=len(itos_l),
    pad_idx=pad_idx
).to(DEVICE)

model.load_state_dict(ckpt["state_dict"])
model.eval()


def read_test_jsonl(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            if "tokens" not in ex:
                raise ValueError("Each JSONL line must contain a 'tokens' field.")
            if "labels" in ex:
                ex = {k: v for k, v in ex.items() if k != "labels"}

            orig_tokens = ex["tokens"]
            norm_tokens = [norm_tok(t) for t in orig_tokens]
            items.append((ex, orig_tokens, norm_tokens))
    return items

test_items = read_test_jsonl(TEST_PATH)


class TestDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ex, orig_tokens, norm_tokens = self.items[idx]
        x = torch.tensor([stoi.get(t, unk_idx) for t in norm_tokens], dtype=torch.long)
        return ex, orig_tokens, x

def collate_test(batch):
    exs, orig_tok_lists, xs = zip(*batch)
    lens = torch.tensor([len(x) for x in xs], dtype=torch.long)
    maxlen = int(lens.max())

    xpad = torch.full((len(xs), maxlen), pad_idx, dtype=torch.long)
    for i, x in enumerate(xs):
        xpad[i, :len(x)] = x

    return exs, orig_tok_lists, xpad, lens

loader = DataLoader(
    TestDataset(test_items),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_test
)


with open(OUT_PATH, "w", encoding="utf-8") as out_f:
    with torch.no_grad():
        for exs, orig_tok_lists, xpad, lens in loader:
            xpad = xpad.to(DEVICE)
            logits = model(xpad, lens)
            pred_ids = logits.argmax(-1).cpu()

            for i, (ex, orig_tokens) in enumerate(zip(exs, orig_tok_lists)):
                L = len(orig_tokens)
                labels = [itos_l[int(pid)] for pid in pred_ids[i, :L]]

                out_ex = dict(ex)
                out_ex["tokens"] = orig_tokens
                out_ex["labels"] = labels

                out_f.write(json.dumps(out_ex, ensure_ascii=False) + "\n")

print("Wrote:", OUT_PATH)